![dvd_image](dvd_image.jpg)

A DVD rental company needs your help! They want to figure out how many days a customer will rent a DVD for based on some features and has approached you for help. They want you to try out some regression models which will help predict the number of days a customer will rent a DVD for. The company wants a model which yeilds a MSE of 3 or less on a test set. The model you make will help the company become more efficient inventory planning.

The data they provided is in the csv file `rental_info.csv`. It has the following features:
- `"rental_date"`: The date (and time) the customer rents the DVD.
- `"return_date"`: The date (and time) the customer returns the DVD.
- `"amount"`: The amount paid by the customer for renting the DVD.
- `"amount_2"`: The square of `"amount"`.
- `"rental_rate"`: The rate at which the DVD is rented for.
- `"rental_rate_2"`: The square of `"rental_rate"`.
- `"release_year"`: The year the movie being rented was released.
- `"length"`: Lenght of the movie being rented, in minuites.
- `"length_2"`: The square of `"length"`.
- `"replacement_cost"`: The amount it will cost the company to replace the DVD.
- `"special_features"`: Any special features, for example trailers/deleted scenes that the DVD also has.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"`: These columns are dummy variables of the rating of the movie. It takes the value 1 if the move is rated as the column name and 0 otherwise. For your convinience, the reference dummy has already been dropped.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Import any additional modules and start coding below

In [185]:
df = pd.read_csv('rental_info.csv')
df.head()

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2
0,2005-05-25 02:54:33+00:00,2005-05-28 23:40:33+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
1,2005-06-15 23:19:16+00:00,2005-06-18 19:24:16+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
2,2005-07-10 04:27:45+00:00,2005-07-17 10:11:45+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
3,2005-07-31 12:06:41+00:00,2005-08-02 14:30:41+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
4,2005-08-19 12:30:04+00:00,2005-08-23 13:35:04+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401


In [186]:
# Convert the 'return_date' and 'rental_date' columns to datetime
df['return_date'] = pd.to_datetime(df['return_date'])
df['rental_date'] = pd.to_datetime(df['rental_date'])

# Calculate the rental length in days
df['rental_length_days'] = (df['return_date'] - df['rental_date']).dt.days


In [187]:
df['deleted_scenes'] = (df['special_features']=='Deleted Scences').astype(int)
df['behind_the_scenes'] =(df['special_features']=='Behind the Scenes').astype(int)

In [188]:
   
X = df.drop(columns=["special_features",'rental_date', 'return_date','rental_length_days'])
y= df['rental_length_days']

print("Length of X:", len(X))
print("Length of y:", len(y))

X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

print("Type of y:", type(y))
print("Shape of y before splitting:", X.shape)

Length of X: 15861
Length of y: 15861
Type of y: <class 'pandas.core.series.Series'>
Shape of y before splitting: (15861, 14)


In [189]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=9)
print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)

Shape of X_train: (12688, 14)
Shape of y_train: (12688,)


In [190]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error as MSE
# from sklearn.preprocessing import LabelEncoder
import numpy as np


# # Convert categorical features to numerical values
# label_encoders = {}
# for column in X_train.select_dtypes(include=['object']).columns:
#     le = LabelEncoder()
#     X_train[column] = le.fit_transform(X_train[column])
#     label_encoders[column] = le

models = [
    ("Linear Regression", LinearRegression()),
    ("Decision Tree", DecisionTreeRegressor()),
    ("Random Forest", RandomForestRegressor())
    
]
best_model = None
best_mse = float('inf')
for name, model in models:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = MSE(y_test, y_pred)
    print(f"{name}:")
    print(f"  Mean Squared Error: {mse}")
    if mse < best_mse:
        best_mse = mse  # Update best MSE
        best_model = model  # Save the best model
print(f"{best_model}:{best_mse}")

Linear Regression:
  Mean Squared Error: 2.9435149213542267
Decision Tree:
  Mean Squared Error: 2.2064469497581958
Random Forest:
  Mean Squared Error: 2.0439286363321805
RandomForestRegressor():2.0439286363321805
